In [ ]:
#Part A — CLIP Validation

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm

from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import spearmanr

PROJECT_ROOT = Path("/Users/yongyili/urban-emotional-intelligence")
DATA_DIR = PROJECT_ROOT / "data"
IMAGE_DIR = PROJECT_ROOT / "images"
ANCHOR_DIR = PROJECT_ROOT / "anchors"

ANCHOR_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Anchor folder:", ANCHOR_DIR)

Project root: /Users/yongyili/urban-emotional-intelligence
Anchor folder: /Users/yongyili/urban-emotional-intelligence/anchors


In [2]:
sentiment_path = DATA_DIR / "sentiment_scores.parquet"

if not sentiment_path.exists():
    raise FileNotFoundError("data/sentiment_scores.parquet not found. Finish Stage 4 first.")

df = pd.read_parquet(sentiment_path)

print("Loaded sentiment scores:", df.shape)
df.head()

Loaded sentiment scores: (655, 42)


,image_id,filepath,borough,road_type,sample_latitude,sample_longitude,latitude,longitude,compass_angle,captured_at,...,scene_description,vader_neg,vader_neu,vader_pos,vader_compound,roberta_neg,roberta_neu,roberta_pos,vader_negativity,negativity_index
0,2567577653535954,images/Kingston_upon_Thames/2567577653535954.jpg,Kingston upon Thames,unknown,51.351090,-0.305450,51.351651,-0.312890,13.042664,1499686461379,...,A road intersection with traffic lights and a ...,0.00,1.00,0.0,0.0000,0.035639,0.788947,0.175414,0.5000,0.267819
1,227868632440032,images/Kingston_upon_Thames/227868632440032.jpg,Kingston upon Thames,unknown,51.401414,-0.262992,51.403125,-0.261115,277.307770,1414503548000,...,A tree-lined paved path with a grassy verge an...,0.11,0.89,0.0,-0.2732,0.022902,0.918193,0.058904,0.6366,0.329751
2,1101120203722336,images/Kingston_upon_Thames/1101120203722336.jpg,Kingston upon Thames,unknown,51.348385,-0.328315,51.351438,-0.330623,58.696777,1545651013361,...,"A multi-lane highway with cars driving on it, ...",0.00,1.00,0.0,0.0000,0.007137,0.614273,0.378590,0.5000,0.253569
3,220316326204880,images/Kingston_upon_Thames/220316326204880.jpg,Kingston upon Thames,unknown,51.386792,-0.270959,51.387736,-0.271491,44.678802,1612180724046,...,A multi-story red brick building with white tr...,0.00,1.00,0.0,0.0000,0.040797,0.817988,0.141216,0.5000,0.270398
4,160502959355506,images/Kingston_upon_Thames/160502959355506.jpg,Kingston upon Thames,unknown,51.391711,-0.310473,51.392385,-0.310321,213.639465,1606489867752,...,"A street scene with buildings on both sides, a...",0.00,1.00,0.0,0.0000,0.034017,0.883162,0.082822,0.5000,0.267008


In [3]:
required_cols = [
    "image_id", "filepath", "borough",
    "stress", "calm", "vitality", "enclosure", "safety", "desolation"
]

missing = [c for c in required_cols if c not in df.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

print("Required columns present.")

Required columns present.


In [4]:
anchor_files = [
    "park1.jpg",
    "quiet_street.jpeg",
    "riverside.jpg",
    "busy_junction.jpg",
    "industrial.jpg",
    "construction.jpg"
]

for f in anchor_files:
    path = ANCHOR_DIR / f
    print(f, path.exists())

park1.jpg True
quiet_street.jpeg True
riverside.jpg True
busy_junction.jpg True
industrial.jpg True
construction.jpg True


In [5]:
import torch
import clip

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ViT-B/32 is lighter and safer on Mac CPU.
# ViT-L/14 is heavier and slower.
model, preprocess = clip.load("ViT-B/32", device=device)

print("CLIP model loaded.")

Device: cpu


100%|███████████████████████████████████████| 338M/338M [00:51<00:00, 6.93MiB/s]


CLIP model loaded.


In [7]:
def get_clip_embedding(filepath):
    filepath = Path(filepath)

    if not filepath.exists():
        raise FileNotFoundError(filepath)

    img = preprocess(Image.open(filepath).convert("RGB")).unsqueeze(0).to(device)

    with torch.no_grad():
        emb = model.encode_image(img)
        emb = emb / emb.norm(dim=-1, keepdim=True)

    return emb.cpu().numpy().squeeze()

In [8]:
test_img_path = PROJECT_ROOT / df.iloc[0]["filepath"]

emb = get_clip_embedding(test_img_path)

print("Embedding shape:", emb.shape)
print("First 5 values:", emb[:5])

Embedding shape: (512,)
First 5 values: [-0.06074274  0.02796346  0.00092375  0.02076988  0.0345743 ]


In [9]:
#compute embeddings for all 655 images
embedding_path = DATA_DIR / "clip_image_embeddings.npy"
embedding_ids_path = DATA_DIR / "clip_image_ids.csv"

if embedding_path.exists() and embedding_ids_path.exists():
    print("Loading existing CLIP embeddings...")
    embeddings = np.load(embedding_path)
    embedding_ids = pd.read_csv(embedding_ids_path)
else:
    print("Computing CLIP embeddings...")
    embeddings_list = []
    image_ids = []

    for _, row in tqdm(df.iterrows(), total=len(df)):
        img_path = PROJECT_ROOT / row["filepath"]
        emb = get_clip_embedding(img_path)
        embeddings_list.append(emb)
        image_ids.append(str(row["image_id"]))

    embeddings = np.vstack(embeddings_list)

    np.save(embedding_path, embeddings)
    pd.DataFrame({"image_id": image_ids}).to_csv(embedding_ids_path, index=False)

print("Embeddings shape:", embeddings.shape)

Computing CLIP embeddings...


  0%|          | 0/655 [00:00<?, ?it/s]

Embeddings shape: (655, 512)


In [11]:
#compute anchor embeddings
calm_anchor_paths = [
    ANCHOR_DIR / "park1.jpg",
    ANCHOR_DIR / "quiet_street.jpeg",
    ANCHOR_DIR / "riverside.jpg"
]

stress_anchor_paths = [
    ANCHOR_DIR / "busy_junction.jpg",
    ANCHOR_DIR / "industrial.jpg",
    ANCHOR_DIR / "construction.jpg"
]

calm_anchor_embs = np.vstack([get_clip_embedding(p) for p in calm_anchor_paths])
stress_anchor_embs = np.vstack([get_clip_embedding(p) for p in stress_anchor_paths])

calm_anchor_mean = calm_anchor_embs.mean(axis=0)
calm_anchor_mean = calm_anchor_mean / np.linalg.norm(calm_anchor_mean)

stress_anchor_mean = stress_anchor_embs.mean(axis=0)
stress_anchor_mean = stress_anchor_mean / np.linalg.norm(stress_anchor_mean)

print("Anchor embeddings ready.")

Anchor embeddings ready.


In [12]:
#compute CLIP similarity scores
df["clip_calm_sim"] = cosine_similarity(embeddings, calm_anchor_mean.reshape(1, -1)).squeeze()
df["clip_stress_sim"] = cosine_similarity(embeddings, stress_anchor_mean.reshape(1, -1)).squeeze()

df[["image_id", "borough", "calm", "stress", "clip_calm_sim", "clip_stress_sim"]].head()

,image_id,borough,calm,stress,clip_calm_sim,clip_stress_sim
0,2567577653535954,Kingston upon Thames,6.1,3.9,0.608550,0.699255
1,227868632440032,Kingston upon Thames,7.0,2.7,0.732120,0.626992
2,1101120203722336,Kingston upon Thames,4.5,4.9,0.546203,0.624283
3,220316326204880,Kingston upon Thames,3.8,6.4,0.597023,0.727595
4,160502959355506,Kingston upon Thames,3.8,6.4,0.548119,0.669446


In [13]:
df[["clip_calm_sim", "clip_stress_sim"]].describe().round(3)

,clip_calm_sim,clip_stress_sim
count,655.000,655.000
mean,0.618,0.670
std,0.076,0.056
min,0.330,0.445
25%,0.571,0.636
50%,0.619,0.678
75%,0.667,0.707
max,0.810,0.813


In [14]:
#CLIP validation correlations
clip_pairs = [
    ("calm", "clip_calm_sim"),
    ("stress", "clip_stress_sim"),
    ("calm", "clip_stress_sim"),
    ("stress", "clip_calm_sim"),
]

clip_results = []

for score_col, clip_col in clip_pairs:
    temp = df[[score_col, clip_col]].dropna()
    r, p = spearmanr(temp[score_col], temp[clip_col])

    clip_results.append({
        "score": score_col,
        "clip_similarity": clip_col,
        "spearman_r": r,
        "p_value": p
    })

clip_results_df = pd.DataFrame(clip_results)
clip_results_df

,score,clip_similarity,spearman_r,p_value
0,calm,clip_calm_sim,0.509981,1.184459e-44
1,stress,clip_stress_sim,0.352792,1.244522e-20
2,calm,clip_stress_sim,-0.241636,3.717362e-10
3,stress,clip_calm_sim,-0.421923,1.170799e-29


In [15]:
clip_results_df.to_csv(DATA_DIR / "clip_validation_results.csv", index=False)

print("Saved:", DATA_DIR / "clip_validation_results.csv")

Saved: /Users/yongyili/urban-emotional-intelligence/data/clip_validation_results.csv


In [16]:
# Keep IDs/path columns safe
string_cols = [
    "image_id", "filepath", "borough", "road_type", "source",
    "scene_description", "greenery_density", "building_height_ratio",
    "traffic_intensity", "pedestrian_activity", "surface_condition", "lighting"
]

for col in string_cols:
    if col in df.columns:
        df[col] = df[col].astype("string")

df.to_parquet(DATA_DIR / "validated_scores.parquet", index=False)
df.to_csv(DATA_DIR / "validated_scores.csv", index=False)

print("Saved validated_scores.parquet and validated_scores.csv")
print("Rows:", len(df))

Saved validated_scores.parquet and validated_scores.csv
Rows: 655


In [ ]:
#Part B — Human Rating Study

In [17]:
#select 40 images for Google Form
scores = pd.read_parquet(DATA_DIR / "validated_scores.parquet")

# 20 random images
random_20 = scores.sample(n=20, random_state=42).copy()
random_20["sample_group"] = "random"

# 20 extreme/diverse images
extreme_parts = []

high_stress = scores.nlargest(5, "stress").copy()
high_stress["sample_group"] = "high_stress"
extreme_parts.append(high_stress)

high_calm = scores.nlargest(5, "calm").copy()
high_calm["sample_group"] = "high_calm"
extreme_parts.append(high_calm)

high_safety = scores.nlargest(5, "safety").copy()
high_safety["sample_group"] = "high_safety"
extreme_parts.append(high_safety)

diverse_boroughs = (
    scores
    .drop_duplicates("borough")
    .sample(n=5, random_state=99)
    .copy()
)
diverse_boroughs["sample_group"] = "diverse_boroughs"
extreme_parts.append(diverse_boroughs)

extreme_20 = pd.concat(extreme_parts, ignore_index=True)

selected = pd.concat([random_20, extreme_20], ignore_index=True)
selected = selected.drop_duplicates(subset="image_id").head(40).copy()

print("Selected images:", len(selected))
print(selected["sample_group"].value_counts())

selected[[
    "image_id", "filepath", "borough", "sample_group",
    "stress", "calm", "safety", "vitality"
]].head()

Selected images: 39
sample_group
random              20
high_stress          5
high_safety          5
diverse_boroughs     5
high_calm            4
Name: count, dtype: int64


,image_id,filepath,borough,sample_group,stress,calm,safety,vitality
0,1048585860586204,images/Redbridge/1048585860586204.jpg,Redbridge,random,5.1,5.5,4.3,4.3
1,379566460086058,images/Ealing/379566460086058.jpg,Ealing,random,5.6,3.2,4.8,4.3
2,1786985018411895,images/Camden/1786985018411895.jpg,Camden,random,4.9,4.5,6.0,5.0
3,322291159395298,images/Bexley/322291159395298.jpg,Bexley,random,4.3,4.8,5.2,3.7
4,266023545929762,images/Richmond_upon_Thames/266023545929762.jpg,Richmond upon Thames,random,4.1,7.2,3.5,2.0


In [18]:
if len(selected) < 40:
    needed = 40 - len(selected)
    extra = scores[~scores["image_id"].isin(selected["image_id"])].sample(n=needed, random_state=123)
    extra["sample_group"] = "extra_fill"
    selected = pd.concat([selected, extra], ignore_index=True)

print("Final selected:", len(selected))

Final selected: 40


In [19]:
selected[[
    "image_id", "filepath", "borough", "sample_group",
    "stress", "calm", "safety", "vitality",
    "scene_description"
]].to_csv(DATA_DIR / "form_selection.csv", index=False)

print("Saved:", DATA_DIR / "form_selection.csv")

Saved: /Users/yongyili/urban-emotional-intelligence/data/form_selection.csv


In [20]:
import shutil

FORM_IMAGE_DIR = PROJECT_ROOT / "human_validation_images"
FORM_IMAGE_DIR.mkdir(exist_ok=True)

for i, (_, row) in enumerate(selected.iterrows(), start=1):
    src = PROJECT_ROOT / row["filepath"]
    dst = FORM_IMAGE_DIR / f"{i:02d}_{row['image_id']}.jpg"
    shutil.copy(src, dst)

print("Copied selected images to:", FORM_IMAGE_DIR)

Copied selected images to: /Users/yongyili/urban-emotional-intelligence/human_validation_images


In [ ]:
#Human rating analysis

In [21]:
form_selection = pd.read_csv(DATA_DIR / "form_selection.csv")
human_raw = pd.read_csv(DATA_DIR / "human_ratings_raw.csv")

print("Form selection:", form_selection.shape)
print("Raw human ratings:", human_raw.shape)

human_raw.head()

Form selection: (40, 9)
Raw human ratings: (4, 161)


,Timestamp,image_01_stress,image_01_calm,image_01_safety,image_01_vitality,image_02_stress,image_02_calm,image_02_safety,image_02_vitality,image_03_stress,...,image_38_safety,image_38_vitality,image_39_stress,image_39_calm,image_39_safety,image_39_vitality,image_40_stress,image_40_calm,image_40_safety,image_40_vitality
0,2026/05/27 8:04:53 pm CET,4,3,2,3,3,4,4,4,2,...,3,2,2,3,3,2,4,2,2,2
1,2026/05/27 10:34:58 pm CET,3,3,4,2,2,3,4,4,2,...,3,4,4,2,3,4,3,2,2,1
2,2026/05/27 10:36:40 pm CET,3,2,2,2,4,3,2,2,3,...,3,3,4,3,2,1,3,3,2,4
3,2026/05/28 5:29:25 pm CET,2,2,3,3,2,1,2,2,1,...,3,3,1,3,4,3,2,2,2,2


In [22]:
rating_rows = []

for rater_idx, row in human_raw.iterrows():
    rater_id = f"rater_{rater_idx+1:02d}"

    for image_num in range(1, 41):
        prefix = f"image_{image_num:02d}"

        stress_col = f"{prefix}_stress"
        calm_col = f"{prefix}_calm"
        safety_col = f"{prefix}_safety"
        vitality_col = f"{prefix}_vitality"

        if stress_col not in human_raw.columns:
            continue

        image_id = form_selection.iloc[image_num - 1]["image_id"]

        rating_rows.append({
            "rater_id": rater_id,
            "image_order": image_num,
            "image_id": str(image_id),
            "stress_h": row[stress_col],
            "calm_h": row[calm_col],
            "safety_h": row[safety_col],
            "vitality_h": row[vitality_col],
        })

human_long = pd.DataFrame(rating_rows)

human_long.to_csv(DATA_DIR / "human_ratings.csv", index=False)

print("Long human ratings:", human_long.shape)
human_long.head()

Long human ratings: (160, 7)


,rater_id,image_order,image_id,stress_h,calm_h,safety_h,vitality_h
0,rater_01,1,1048585860586204,4,3,2,3
1,rater_01,2,379566460086058,3,4,4,4
2,rater_01,3,1786985018411895,2,3,4,5
3,rater_01,4,322291159395298,4,2,3,2
4,rater_01,5,266023545929762,1,4,3,5


In [23]:
human_mean = (
    human_long
    .groupby("image_id")[["stress_h", "calm_h", "safety_h", "vitality_h"]]
    .mean()
    .reset_index()
)

human_mean.head()

,image_id,stress_h,calm_h,safety_h,vitality_h
0,1004577386845832,3.75,1.5,2.25,4.5
1,1013794687253153,3.50,2.0,2.50,3.0
2,1048585860586204,3.00,2.5,2.75,2.5
3,1069554004511711,3.25,2.0,2.50,2.5
4,1103962721751983,2.00,2.5,2.50,1.5


In [24]:
scores = pd.read_parquet(DATA_DIR / "validated_scores.parquet")

scores["image_id"] = scores["image_id"].astype(str)
human_mean["image_id"] = human_mean["image_id"].astype(str)

merged = human_mean.merge(
    scores,
    on="image_id",
    how="left"
)

print("Merged:", merged.shape)
merged[[
    "image_id", "borough",
    "stress_h", "stress",
    "calm_h", "calm",
    "safety_h", "safety",
    "vitality_h", "vitality"
]].head()

Merged: (40, 48)


,image_id,borough,stress_h,stress,calm_h,calm,safety_h,safety,vitality_h,vitality
0,1004577386845832,Kensington and Chelsea,3.75,6.9,1.5,2.2,2.25,5.0,4.5,6.0
1,1013794687253153,Richmond upon Thames,3.50,5.7,2.0,5.2,2.50,5.8,3.0,4.3
2,1048585860586204,Redbridge,3.00,5.1,2.5,5.5,2.75,4.3,2.5,4.3
3,1069554004511711,Hackney,3.25,7.7,2.0,2.9,2.50,4.3,2.5,4.3
4,1103962721751983,Hounslow,2.00,2.4,2.5,6.1,2.50,4.8,1.5,2.3


In [25]:
pairs = [
    ("stress_h", "stress"),
    ("calm_h", "calm"),
    ("safety_h", "safety"),
    ("vitality_h", "vitality")
]

human_corr_rows = []

print("=== Human vs VLM-derived scores: full sample ===")

for h_col, model_col in pairs:
    temp = merged[[h_col, model_col]].dropna()
    r, p = spearmanr(temp[h_col], temp[model_col])

    human_corr_rows.append({
        "sample": "full_40",
        "human_score": h_col,
        "model_score": model_col,
        "spearman_r": r,
        "p_value": p
    })

    print(f"{model_col:10s}: r={r:+.3f}, p={p:.4f}")

=== Human vs VLM-derived scores: full sample ===
stress    : r=+0.814, p=0.0000
calm      : r=+0.578, p=0.0001
safety    : r=+0.678, p=0.0000
vitality  : r=+0.456, p=0.0031


In [26]:
random_ids = form_selection[form_selection["sample_group"] == "random"]["image_id"].astype(str).tolist()

merged_random = merged[merged["image_id"].astype(str).isin(random_ids)]

print("=== Human vs VLM-derived scores: random 20 primary sample ===")

for h_col, model_col in pairs:
    temp = merged_random[[h_col, model_col]].dropna()
    r, p = spearmanr(temp[h_col], temp[model_col])

    human_corr_rows.append({
        "sample": "random_20_primary",
        "human_score": h_col,
        "model_score": model_col,
        "spearman_r": r,
        "p_value": p
    })

    print(f"{model_col:10s}: r={r:+.3f}, p={p:.4f}")

=== Human vs VLM-derived scores: random 20 primary sample ===
stress    : r=+0.758, p=0.0001
calm      : r=+0.646, p=0.0021
safety    : r=+0.578, p=0.0076
vitality  : r=+0.461, p=0.0406


In [28]:
human_corr_df = pd.DataFrame(human_corr_rows)

human_corr_df.to_csv(DATA_DIR / "human_validation_correlations.csv", index=False)
merged.to_csv(DATA_DIR / "human_validation_results.csv", index=False)

print("Saved human validation outputs.")

Saved human validation outputs.


In [29]:
print("A small exploratory human validation study was conducted with 4 respondents rating 40 London street images. Despite the limited number of raters, all four evaluated dimensions showed positive Spearman correlations between human mean ratings and model-derived scores. The random 20-image subset, used as the primary validation sample, showed particularly strong alignment for stress, calm, safety, and vitality. These results suggest that the feature-derived scoring method captures perceptual patterns that are broadly consistent with human judgements, while remaining limited by the small rater sample.")

A small exploratory human validation study was conducted with 4 respondents rating 40 London street images. Despite the limited number of raters, all four evaluated dimensions showed positive Spearman correlations between human mean ratings and model-derived scores. The random 20-image subset, used as the primary validation sample, showed particularly strong alignment for stress, calm, safety, and vitality. These results suggest that the feature-derived scoring method captures perceptual patterns that are broadly consistent with human judgements, while remaining limited by the small rater sample.


In [30]:
scores = pd.read_parquet(DATA_DIR / "validated_scores.parquet")
scores[["clip_calm_sim", "clip_stress_sim"]].head()

,clip_calm_sim,clip_stress_sim
0,0.608550,0.699255
1,0.732120,0.626992
2,0.546203,0.624283
3,0.597023,0.727595
4,0.548119,0.669446


In [31]:
human_results = pd.read_csv(DATA_DIR / "human_validation_results.csv")
human_corr = pd.read_csv(DATA_DIR / "human_validation_correlations.csv")

print(human_results.shape)
print(human_corr)

(40, 48)
              sample human_score model_score  spearman_r       p_value
0            full_40    stress_h      stress    0.813523  1.767092e-10
1            full_40      calm_h        calm    0.577589  9.524769e-05
2            full_40    safety_h      safety    0.678443  1.502624e-06
3            full_40  vitality_h    vitality    0.455995  3.105281e-03
4  random_20_primary    stress_h      stress    0.758029  1.078380e-04
5  random_20_primary      calm_h        calm    0.645569  2.110196e-03
6  random_20_primary    safety_h      safety    0.577766  7.631427e-03
7  random_20_primary  vitality_h    vitality    0.461375  4.059480e-02
